# 02A Full Refresh Demo

Run the completed retail example: **Full Read → PySpark transformation → Overwrite**. This notebook preserves the working Orders, Products, and Order History pipeline that previously lived in `02_pipeline.ipynb`.

## Tested with FabricOps

This completed demo preserves the Microsoft Fabric runtime validation record of the full-refresh pipeline it was copied from.

| FabricOps release | Tested by | Date tested |
|---|---|---|
| v0.2.5 | Voyce | 12 Oct 2026 |

The Activated Pipeline lifecycle has local structural/control-flow validation only; the historical Fabric records above describe the prior scaffold. Actual Fabric verification is still required.


# 0. Environment

Run the shared Fabric configuration and import the public APIs used by this pipeline.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import finalize_pipeline, orchestrate_read, orchestrate_write, setup_pipeline

# 1. Pipeline Data Contracts

Keep the notebook executable unchanged across Testing and Live. Set runtime controls in `00_env_config` or notebook parameters: `PIPELINE_CONTRACT_TESTING=True` enables the Testing inspector. The default is unattended **Bootstrap**; it creates initial outputs only while no participant has a frozen contract.

After Governance freezes every source/target contract, run complete Testing and call `finalize_pipeline()` after every declared operation. Testing persists run evidence and Guardrail diagnostics while business publication and ordinary operational metadata remain disabled. Stopped notebooks remain incomplete.

Governance then activates the exact successful whole pipeline. Package its approved manifest, exact definitions and exported notebook; deploy that unchanged executable; verify its workspace, notebook and store bindings. Configure `PIPELINE_LIVE_ENABLED=True` and `PIPELINE_DEPLOYMENT_PATH` externally. Supply `PIPELINE_SELECT_ACTIVATED_VERSION` once to save the approved version, then omit it for unattended manual/scheduled Live runs. Saved selections never switch silently.

See the [complete lifecycle and migration guide](https://voycepeh.github.io/FabricOps-Starter-Kit/guided-demo/05-activate-data-contract-and-promote/). Live rejects missing approval, changed executable/dependencies/configuration, invalid bindings and manual contract overrides; it never falls back to Bootstrap.


In [ ]:
# Default first run: no UI and no contract selections. Enable after contracts are frozen.
CONTRACT_TESTING = bool(globals().get("PIPELINE_CONTRACT_TESTING", False))
PIPELINE_LIVE = bool(globals().get("PIPELINE_LIVE_ENABLED", False))
DEPLOYMENT_PATH = globals().get("PIPELINE_DEPLOYMENT_PATH")
SELECT_ACTIVATED_VERSION = globals().get("PIPELINE_SELECT_ACTIVATED_VERSION")

# Declare actual participants before their targets or Catalogue records exist.
PIPELINE_SOURCES = [{'store': 'Bronze', 'schema': 'demo', 'table_name': 'orders'},
 {'store': 'Bronze', 'schema': 'demo', 'table_name': 'products'},
 {'store': 'Gold', 'schema': 'demo', 'table_name': 'order_history'}]
PIPELINE_TARGETS = [{'store': 'Silver', 'schema': 'demo', 'table_name': 'curated_orders'},
 {'store': 'Gold', 'schema': 'demo', 'table_name': 'customer_summary'}]
# Replace with your unique project pipeline name; preserve across environments and runs.
PIPELINE_ID = "02A_full_refresh_demo"
# Fabric verifies the exact deployed notebook definition. An unavailable identity
# leaves Testing incomplete. Keep transformation/helper code in the notebook.
EXECUTABLE_SHA256 = None
# Pin extra project libraries here; {} declares none beyond the tracked runtime.
PROJECT_DEPENDENCIES = {}
PROJECT_CONFIGURATION = globals().get("PIPELINE_PROJECT_CONFIGURATION", {})
CONTRACTS = setup_pipeline(
    pipeline_id=PIPELINE_ID, executable_sha256=EXECUTABLE_SHA256,
    dependencies=PROJECT_DEPENDENCIES, configuration=PROJECT_CONFIGURATION,
    sources=PIPELINE_SOURCES, targets=PIPELINE_TARGETS,
    execution_mode="live" if PIPELINE_LIVE else None,
    interactive=CONTRACT_TESTING and not PIPELINE_LIVE,
    deployment_path=DEPLOYMENT_PATH if PIPELINE_LIVE else None,
    activated_pipeline_id=SELECT_ACTIVATED_VERSION if PIPELINE_LIVE else None,
    spark_session=spark,
)


In [ ]:
# Bootstrap already returns its plan. Testing builds after explicit widget selections.
EXECUTION_PLAN = CONTRACTS["build_plan"]() if CONTRACT_TESTING and not PIPELINE_LIVE else CONTRACTS


# 2. Read

`orchestrate_read()` is the plug-and-play governed source read. You provide the source location and read behaviour; FabricOps runs the standard source-side lifecycle underneath and returns the DataFrame plus the results of each check.

**Under the hood**

- **Read** — read the governed source.
- **Freshness** — check the source freshness expectation.
- **Schema** — validate the observed schema against the Data Contract.
- **Grain** — evaluate the configured Grain Guardrail.
- **Data Quality** — evaluate configured DQ rules.
- **Profile** — profile the full source for governance and inspection.

**Parameters**

- The `sources` dictionary key — notebook-facing label, not an orchestrator argument.
- `store` — configured FabricStore key from `00_env_config` such as `Bronze`, `Silver`, or `Gold`.
- `schema` — physical Lakehouse or Warehouse schema containing the source table.
- `table_name` — physical source table name.
- `read_mode` — `full` for the complete source or `incremental` for target-aware incremental processing.
- `query` — optional read-only Warehouse SQL pushdown. Use `None` for a normal table read.
- `read_parameters` — Incremental options such as `watermark_column`; not needed for these Full reads.
- `target_table_id` — required for Incremental reads so FabricOps can resolve target-aware state.
- `grain_discovery_enabled`, `grain_discovery_max_columns`, `grain_discovery_max_candidates` — control candidate Grain profiling, not contract Grain checks.
- `frequency_enabled`, `frequency_top_n` — control Frequency profiling.
- `dq_diagnostics` — `"sample"` (default), `"none"` (metrics only), or `"full"` (lazy failure rows).
- `verbose` — enable stage and final summaries.
- `spark_session` — active Fabric Spark session.

The returned object includes `dataframe`, `table_id`, source check results, profiling results, and orchestration stage metadata. Optional inspection lines are kept commented out so normal runs stay clean.

Clone a Read block for each governed source.

In [ ]:
# Dictionary used to keep source results for transformation and lineage.
sources = {}

## READ 1 — Orders

In [ ]:
source = orchestrate_read(
    execution_plan=EXECUTION_PLAN,
    store="Bronze",
    schema="demo",
    table_name="orders",
    read_mode="full",
    query=None,
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["orders"] = source
# Inspect the actual rows returned by this Read.
display(sources["orders"]["dataframe"].limit(10))


## READ 2 — Products

In [ ]:
source = orchestrate_read(
    execution_plan=EXECUTION_PLAN,
    store="Bronze",
    schema="demo",
    table_name="products",
    read_mode="full",
    query=None,
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["products"] = source
# Inspect the actual rows returned by this Read.
display(sources["products"]["dataframe"].limit(10))


## READ 3 — Order History

In [ ]:
source = orchestrate_read(
    execution_plan=EXECUTION_PLAN,
    store="Gold",
    schema="demo",
    table_name="order_history",
    read_mode="full",
    query="""
SELECT
    historical_order_id,
    customer_id,
    order_datetime,
    net_amount
FROM demo.order_history
""",
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["history"] = source
# Inspect the actual rows returned by this Read.
display(sources["history"]["dataframe"].limit(10))


# 3. PySpark Transform

Use the named source DataFrames to build the DataFrames you want to publish. This example produces two different outputs so the WRITE blocks demonstrate publishing different tables.

In [ ]:
# Pull the source DataFrames back out of the sources dictionary by name.
orders_df = sources["orders"]["dataframe"]
products_df = sources["products"]["dataframe"]
history_df = sources["history"]["dataframe"]

history_summary_df = (
    history_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("historical_order_count"),
        F.sum("net_amount").alias("historical_net_amount"),
        F.max("order_datetime").alias("latest_historical_order_datetime"),
    )
)

# Output 1: detailed order-level table.
transformed_df = (
    orders_df.alias("orders")
    .join(products_df.alias("products"), on="product_id", how="left")
    .join(history_summary_df.alias("history"), on="customer_id", how="left")
    .withColumn("order_net_amount", F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2))
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)

# Output 2: customer-level summary derived from the detailed order output.
customer_summary_df = (
    transformed_df
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("order_net_amount").alias("total_order_net_amount"),
        F.max("order_datetime").alias("latest_order_datetime"),
        F.max("historical_order_count").alias("historical_order_count"),
        F.max("historical_net_amount").alias("historical_net_amount"),
    )
)

# Optional development inspection — uncomment only when needed.
# Detailed order-level transformed output that will feed WRITE 1.
# display(transformed_df)
# Customer-level aggregated output that will feed WRITE 2.
# display(customer_summary_df)

# 4. Write

`orchestrate_write()` is the plug-and-play governed target write. You pass the transformed DataFrame, its governed sources, destination, and load strategy; FabricOps resolves the target contract, runs target-side guardrails, and only publishes when the shared execution plan is Live and the checks permit continuation.

**Under the hood**

- **Schema** — validate the target schema against the Data Contract.
- **Sensitive Data** — detect and apply configured sensitive-data treatment.
- **Source Drift** — check governed source changes against target expectations.
- **Grain** — evaluate the configured target Grain Guardrail.
- **Data Quality** — evaluate configured DQ rules.
- **Write** — publish the target when running in Live mode.
- **Profile** — profile the persisted target after a successful write.

In **Testing**, every declared Write evaluates its prepared input and returns diagnostics without target publication or ordinary metadata commits. In **Live**, successful checks continue through publication and persisted-target profiling. Inspect each target result; an unpublished result does not imply its checks passed.

**Parameters**

- first argument — transformed Spark DataFrame to validate and publish.
- The `writes` dictionary key — notebook-facing label, not an orchestrator argument.
- `sources` — the `orchestrate_read()` result objects that contributed to this target; their `table_id` values drive lineage and source-side contract checks.
- `store` — configured destination FabricStore key from `00_env_config`.
- `schema` — physical target schema.
- `table_name` — physical target table name.
- `write_mode` — governed write strategy such as `overwrite`, `append`, `scd1`, or `scd2`.
- `write_parameters` — mode-specific options, such as SCD keys and effective dates.
- `execution_plan` — immutable plan built through `setup_pipeline()`; pass the same plan to every Read and Write for one Testing or Live run.
- `repartition_by` — optional Spark partition count before the write.
- `grain_discovery_enabled`, `grain_discovery_max_columns`, `grain_discovery_max_candidates` — control candidate Grain profiling, not contract Grain checks.
- `frequency_enabled`, `frequency_top_n` — control Frequency profiling.
- `dq_diagnostics` — `"sample"` (default), `"none"` (metrics only), or `"full"` (lazy failure rows).
- `existing_mapping` — optional caller-managed Sensitive Data token mapping; omit when a managed mapping store is configured.
- `verbose` — enable stage and final summaries.
- `spark_session` — active Fabric Spark session.

The returned object includes the target `table_id`, guardrail results, publication status, orchestration stage metadata, and, when published, persisted-target profiling results. Optional inspection lines are kept commented out so normal runs stay clean.

Clone a Write block for each governed target.

In [ ]:
# Dictionary used to keep publication results for later reference.
writes = {}

## WRITE 1 — Lakehouse Curated Orders

In [ ]:
write_result = orchestrate_write(
    transformed_df,
    sources=[sources["orders"], sources["products"], sources["history"]],
    store="Silver",
    schema="demo",
    table_name="curated_orders",
    write_mode="overwrite",
    execution_plan=EXECUTION_PLAN,
    repartition_by=None,
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
writes["curated_orders_lakehouse"] = write_result
# Inspect the DataFrame submitted to this Write.
display(write_result["dataframe"].limit(10))

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(write_result.get("dq_result"))
# display((write_result.get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the persisted target after a successful Live publication.
# from fabricops_kit import read_lakehouse_table
# display(read_lakehouse_table(store="Silver", schema="demo", table_name="curated_orders", spark_session=spark).limit(10))


## WRITE 2 — Warehouse Customer Summary

In [ ]:
write_result = orchestrate_write(
    customer_summary_df,
    sources=[sources["orders"], sources["products"], sources["history"]],
    store="Gold",
    schema="demo",
    table_name="customer_summary",
    write_mode="overwrite",
    execution_plan=EXECUTION_PLAN,
    repartition_by=4,
    grain_discovery_enabled=True,  # False skips candidate profiling, not Grain Guardrails.
    grain_discovery_max_columns=3,
    grain_discovery_max_candidates=8,
    frequency_enabled=True,  # Set False to skip frequency distributions.
    frequency_top_n=5,
    spark_session=spark,
    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
writes["customer_summary_warehouse"] = write_result
# Inspect the DataFrame submitted to this Write.
display(write_result["dataframe"].limit(10))

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(write_result.get("dq_result"))
# display((write_result.get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the persisted target after a successful Live publication.
# from fabricops_kit import read_warehouse_query
# display(read_warehouse_query("SELECT TOP 10 * FROM demo.customer_summary", store="Gold", spark_session=spark))


In [ ]:
# Complete the logical run only after all declared operations and transforms.
PIPELINE_RUN = finalize_pipeline(EXECUTION_PLAN, spark_session=spark)
print("Pipeline run status:", PIPELINE_RUN["status"])

# Report publication outcomes and refresh contract testing results., including failed Testing checks.
for target_name, result in writes.items():
    print(target_name, "mode:", result["execution_mode"],
          "publication:", result["publication_status"],
          "validation_passed:", result.get("validation_passed", "not_applicable"))

# Refresh expandable Data Contract sections with this plan's execution results.
if CONTRACT_TESTING and not PIPELINE_LIVE:
    CONTRACTS["refresh"]()
